# L1 and L2 Regularization

Implement L1 and L2 penalty terms from scratch, add them to a regression loss, and observe how each penalty shapes the learned weights: L1 promotes sparsity, L2 promotes uniform shrinkage.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


## Regularizer vs loss: an important distinction

**Loss** (MSE, MAE, cross-entropy): measures how well predictions match targets. It defines the training objective and corresponds to a probabilistic likelihood.

**Regularizer** (L1, L2 penalty): adds a constraint term to the loss to control weight magnitude. The regularizer penalizes model complexity and is unrelated to how predictions compare to labels.

The combined objective is:

$$J(\theta) = L(\theta) + \lambda \cdot \Omega(\theta)$$

where \(L\) is the task loss, \(\Omega\) is the regularizer, and \(\lambda\) is the penalty strength.

In [2]:
torch.manual_seed(0)
N = 200
D = 20  # 20 features; only first 5 are truly informative

# True weights: sparse — only first 5 features matter
true_w = torch.zeros(D, device=device)
true_w[:5] = torch.tensor([3.0, -2.0, 1.5, -1.0, 0.5], device=device)

X = torch.randn(N, D, device=device)
y = X @ true_w + 0.2 * torch.randn(N, device=device)

# Train/val split
X_tr, X_val = X[:160], X[160:]
y_tr, y_val = y[:160], y[160:]
print(f"Train: {X_tr.shape}, Val: {X_val.shape}")
print(f"True weights (first 10): {true_w[:10].tolist()}")


Train: torch.Size([160, 20]), Val: torch.Size([40, 20])
True weights (first 10): [3.0, -2.0, 1.5, -1.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0]


## L1 and L2 penalties from scratch

$$\Omega_{L1}(w) = \|w\|_1 = \sum_j |w_j|$$

$$\Omega_{L2}(w) = \|w\|_2^2 = \sum_j w_j^2$$

Note: the conventional L2 penalty uses the **squared** \(\ell_2\) norm, not the norm itself. This makes the gradient smooth: \(\nabla_w \Omega_{L2} = 2w\). The L1 penalty has subgradient \(\text{sign}(w)\), which is non-smooth at zero — this kink is what allows exact zeros and therefore sparsity.

In [3]:
def l1_penalty(w: torch.Tensor) -> torch.Tensor:
    """L1 penalty: sum of absolute values of weights."""
    return w.abs().sum()


def l2_penalty(w: torch.Tensor) -> torch.Tensor:
    """L2 penalty: sum of squared weights (squared L2 norm)."""
    return (w ** 2).sum()


def mse_loss(w: torch.Tensor, X: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Mean squared error: (1/N) sum_i (X_i @ w - y_i)^2."""
    return ((X @ w - y) ** 2).mean()


# Quick sanity checks
w_test = torch.tensor([1.0, -2.0, 0.0, 3.0], device=device)
print(f"L1 penalty on [1,-2,0,3]: {l1_penalty(w_test).item():.1f}  (expected 6.0)")
print(f"L2 penalty on [1,-2,0,3]: {l2_penalty(w_test).item():.1f}  (expected 14.0)")
assert l1_penalty(w_test).item() == 6.0
assert l2_penalty(w_test).item() == 14.0
print("Penalty sanity checks ✓")


L1 penalty on [1,-2,0,3]: 6.0  (expected 6.0)
L2 penalty on [1,-2,0,3]: 14.0  (expected 14.0)
Penalty sanity checks ✓


## Validating the L2 gradient against autograd

The analytic gradient of the L2 penalty with respect to \(w\) is \(2\lambda w\). We verify this against `torch.autograd`.

In [4]:
lam = 0.1
w_check = torch.randn(D, device=device, requires_grad=True)

# Autograd gradient of lambda * L2(w)
penalty = lam * l2_penalty(w_check)
penalty.backward()
autograd_grad = w_check.grad.clone()

# Analytic gradient: 2 * lambda * w
analytic_grad = 2 * lam * w_check.detach()

print("Max absolute difference (autograd vs 2*lambda*w):", (autograd_grad - analytic_grad).abs().max().item())
assert torch.allclose(autograd_grad, analytic_grad, atol=1e-6), (
    f"L2 gradient mismatch: max diff {(autograd_grad - analytic_grad).abs().max().item()}"
)
print("L2 gradient matches 2*lambda*w ✓")


Max absolute difference (autograd vs 2*lambda*w): 0.0
L2 gradient matches 2*lambda*w ✓


## Training linear regression with L1 / L2 regularization

We train a linear model under varying penalty strengths and inspect the learned weights. L1 should drive many weights toward zero (sparsity); L2 should shrink all weights uniformly.

In [5]:
def train_with_penalty(
    penalty_fn,
    lam: float,
    steps: int = 800,
    lr: float = 0.01,
) -> torch.Tensor:
    """Train linear regression with a given penalty function.

    Uses immutable-update SGD (create a new tensor each step) for
    clarity. Returns the final weight vector on CPU.
    """
    w = torch.zeros(D, device=device, requires_grad=True)
    for _ in range(steps):
        loss = mse_loss(w, X_tr, y_tr) + lam * penalty_fn(w)
        loss.backward()
        # Immutable update: create new tensor, do not mutate w
        w = (w - lr * w.grad).detach().requires_grad_(True)
    return w.detach().cpu()


lambdas = [0.0, 0.01, 0.1, 1.0]
results_l1 = {lam: train_with_penalty(l1_penalty, lam) for lam in lambdas}
results_l2 = {lam: train_with_penalty(l2_penalty, lam) for lam in lambdas}
print("Training complete for L1 and L2 at lambdas:", lambdas)


Training complete for L1 and L2 at lambdas: [0.0, 0.01, 0.1, 1.0]


In [6]:
feature_idx = list(range(D))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for lam, w_learned in results_l1.items():
    axes[0].plot(feature_idx, w_learned.abs().tolist(), marker="o", ms=4, label=f"λ={lam}")
axes[0].set_title("L1 penalty — weight magnitudes")
axes[0].set_xlabel("feature index")
axes[0].set_ylabel("|w_j|")
axes[0].axvline(x=4.5, color="gray", linestyle="--", label="informative | noise boundary")
axes[0].legend(fontsize=8)

for lam, w_learned in results_l2.items():
    axes[1].plot(feature_idx, w_learned.abs().tolist(), marker="o", ms=4, label=f"λ={lam}")
axes[1].set_title("L2 penalty — weight magnitudes")
axes[1].set_xlabel("feature index")
axes[1].set_ylabel("|w_j|")
axes[1].axvline(x=4.5, color="gray", linestyle="--", label="informative | noise boundary")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()
print("""
L1 (left): increasing lambda drives noise features sharply toward 0 — sparsity.
L2 (right): weights shrink proportionally but remain nonzero — shrinkage.
""")



L1 (left): increasing lambda drives noise features sharply toward 0 — sparsity.
L2 (right): weights shrink proportionally but remain nonzero — shrinkage.



/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_17168/987654141.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
print("Near-zero weights (|w_j| < 0.05) per lambda:")
print(f"{'lambda':>8} | {'L1 sparse':>10} | {'L2 sparse':>10}")
for lam in lambdas:
    n_l1 = (results_l1[lam].abs() < 0.05).sum().item()
    n_l2 = (results_l2[lam].abs() < 0.05).sum().item()
    print(f"{lam:>8.3f} | {n_l1:>10} | {n_l2:>10}")
print()
# For strong L1, noise features should be much sparser than for L2
strong_l1_sparse = (results_l1[1.0].abs() < 0.05).sum().item()
strong_l2_sparse = (results_l2[1.0].abs() < 0.05).sum().item()
assert strong_l1_sparse > strong_l2_sparse, (
    f"Expected L1 to produce more zeros than L2 at lambda=1.0: L1={strong_l1_sparse}, L2={strong_l2_sparse}"
)
print("L1 produces more near-zero weights than L2 at strong regularization ✓")


Near-zero weights (|w_j| < 0.05) per lambda:
  lambda |  L1 sparse |  L2 sparse
   0.000 |         15 |         15
   0.010 |         15 |         15
   0.100 |         15 |         13
   1.000 |         15 |          6

L1 produces more near-zero weights than L2 at strong regularization ✓


## L2 penalty vs weight decay

Adding an L2 penalty \(\lambda \|w\|_2^2\) to the loss is equivalent to *coupled* weight decay: the update becomes

$$w \leftarrow w - \alpha \nabla_w L(w) - 2\alpha\lambda w$$

Standard SGD treats this identically to weight decay. However, adaptive optimizers like Adam *scale* gradient updates by the second moment, which also scales the L2 gradient term. This means L2-penalized Adam behaves differently from AdamW, where weight decay is applied to the parameter *before* the gradient scaling step (decoupled weight decay). For modern deep learning, prefer AdamW over Adam+L2.

In [8]:
# Demonstrate: gradient of (MSE + lambda*L2) equals gradient of MSE + 2*lambda*w
w_demo = torch.randn(D, device=device, requires_grad=True)
lam_demo = 0.05

# Joint loss
total_loss = mse_loss(w_demo, X_tr, y_tr) + lam_demo * l2_penalty(w_demo)
total_loss.backward()
joint_grad = w_demo.grad.clone()

# Separately: gradient of MSE + 2*lambda*w
w_demo2 = w_demo.detach().requires_grad_(True)
mse_only = mse_loss(w_demo2, X_tr, y_tr)
mse_only.backward()
expected_grad = w_demo2.grad + 2 * lam_demo * w_demo2.detach()

assert torch.allclose(joint_grad, expected_grad, atol=1e-5), (
    f"Gradient mismatch: max diff {(joint_grad - expected_grad).abs().max().item()}"
)
print("Gradient of (MSE + lambda*L2) equals grad(MSE) + 2*lambda*w ✓")
print("This is the weight-decay equivalence for SGD.")


Gradient of (MSE + lambda*L2) equals grad(MSE) + 2*lambda*w ✓
This is the weight-decay equivalence for SGD.


## The idiomatic PyTorch way

In practice: pass `weight_decay` to the optimizer. For SGD this is equivalent to L2 regularization. For AdamW it is decoupled weight decay. Avoid manually adding L2 terms to the loss when using adaptive optimizers — use AdamW instead.

In [9]:
# Idiomatic: Adam with weight_decay (decoupled in AdamW, coupled in Adam)
w_torch = torch.zeros(D, device=device, requires_grad=True)
opt = torch.optim.SGD([w_torch], lr=0.01, weight_decay=0.1 * 2)  # SGD weight_decay = 2*lambda
for _ in range(800):
    opt.zero_grad()
    loss = mse_loss(w_torch, X_tr, y_tr)
    loss.backward()
    opt.step()

# Should closely match our from-scratch L2 result for lambda=0.1
scratch_l2 = results_l2[0.1]  # computed earlier on CPU
torch_l2 = w_torch.detach().cpu()
print("Max absolute difference (scratch L2 vs torch SGD+weight_decay):",
      (scratch_l2 - torch_l2).abs().max().item())
assert (scratch_l2 - torch_l2).abs().max().item() < 0.05, (
    "SGD+weight_decay diverges from scratch L2 regularization"
)
print("SGD weight_decay is equivalent to L2 penalty in the loss ✓")


Max absolute difference (scratch L2 vs torch SGD+weight_decay): 0.0
SGD weight_decay is equivalent to L2 penalty in the loss ✓


## Takeaways

- A **regularizer** is a penalty added to the training loss to constrain weight magnitude. It is distinct from the **loss** (MSE, BCE, etc.) which measures prediction error.
- **L1 penalty** (\(\|w\|_1\)) has a kink at zero, so its subgradient can push weights exactly to zero. This produces sparse solutions, making L1 useful for feature selection.
- **L2 penalty** (\(\|w\|_2^2\)) has a smooth gradient \(2\lambda w\), which shrinks all weights proportionally but almost never produces exact zeros.
- The L2 gradient is \(2\lambda w\), not \(\lambda w\) — the factor of 2 comes from differentiating the squared norm.
- For SGD, L2 regularization is equivalent to weight decay. For adaptive optimizers (Adam), they are **not** equivalent — use AdamW for decoupled weight decay.
- Increasing \(\lambda\) increases bias and decreases variance, moving along the bias-variance tradeoff curve. The optimal \(\lambda\) is chosen by cross-validation.